In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

**<font size="6" color="red">ch04 RNN_Recurrent Neural Network</font>**
- 순서나 시간 데이터가 중요할 때 ex. 번역, 음성인식, 주가예측

# 1. 문맥을 이용하여 모델만들기

In [ ]:
text = """경마장에 있는 말이 뛰고 있다
그의 말이 법이다
가는 말이 고와야 오는 말이 곱다"""
text1 = "겨울이 오는 날"

In [ ]:
from keras_preprocessing.text import Tokenizer
t = Tokenizer()
t.fit_on_texts([text, text1])
encoded = t.texts_to_sequences([text, text1])
print(encoded)
print(t.word_index)

In [ ]:
text = """경마장에 있는 말이 뛰고 있다
그의 말이 법이다
가는 말이 고와야 오는 말이 곱다"""

In [ ]:
# 문자를 인덱스시퀀스로 변환하기 위한 과정
t = Tokenizer()
t.fit_on_texts([text])
print(t.word_index)

In [ ]:
# 문자열 리스트를 인덱스 시퀀스로 변환
print(t.texts_to_sequences(['경마장에 말이 있다', '말이 뛴다']))
print(t.texts_to_sequences(['가는 말이 곱다'])[0])

In [ ]:
for key, value in t.word_index.items():
    print(key, value)

In [ ]:
text = """경마장에 있는 말이 뛰고 있다
그의 말이 법이다
가는 말이 고와야 오는 말이 곱다"""

In [ ]:
# text를 학습시키기 위한 ['경마장에 있는', '경마장에 있는 말이', ....]
sequences = []
for line in text.split('\n'):
    print('원 문장 : ',line)
    encoded = t.texts_to_sequences([line])[0]
    print('encoded된 문장 :', encoded)
    for i in range(0, len(encoded)-1): # 시작index
        for j in range(i+2, len(encoded)+1 ): # 끝나는 index
            sequences.append(encoded[i:j])
#sequences
print('sequences와 해석 출력')
for sequence in sequences:
    # print(sequence)
    for word_seq in sequence:
        for key, value in t.word_index.items():
            if word_seq==value:
                print("{}:{}".format(word_seq, key), end=' ')
                break
    print()

In [ ]:
# sequences의 길이를 모두 같게(padding)
my_len = max([len(seq) for seq in sequences])
my_len

In [ ]:
# sequences를 훈련 가능하도록 6개(앞에 0, 뒤에 0)
from tensorflow.keras.preprocessing.sequence import pad_sequences
padded_sequences = pad_sequences(sequences=sequences,
                                maxlen=my_len,
                                padding='pre', # 앞에 0을 padding
                                #truncating='post',
                                )
padded_sequences[:3], padded_sequences.shape

In [ ]:
# 독립변수(X)와 타겟변수(y)를 분리
X = padded_sequences[:, :-1]
y = padded_sequences[:, -1]
# 단어 갯수
vocab_size = len(t.word_index)
vocab_size

In [ ]:
# 종속변수의 원핫인코딩
from tensorflow.keras.utils import to_categorical
Y = to_categorical(y, vocab_size+1)
Y[:3]

In [ ]:
X.shape, Y.shape, vocab_size

In [ ]:
# 모델 생성 ( Embedding -> RNN -> Dense층 )
# Embedding층의 입력은 12(희소행렬) -> 출력은 10. 이 단계에서 필요한 Embedding weight matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
# 교안 106p.
model = Sequential(name='sequential')
model.add(Embedding(input_dim=vocab_size+1, # 임베딩 입력
                   output_dim=10, # 임베딩 출력
                   input_length=X.shape[1])) # 입력(독립)변수 단어 길이
model.add(SimpleRNN(32, activation='tanh')) # 기본 activation='tanh'
model.add(Dense(12, activation='softmax'))
model.summary()

In [ ]:
# 학습과정 설정
model.compile(loss='categorical_crossentropy',
             optimizer='adam',
             metrics=['accuracy'])
# 학습시키기
hist = model.fit(X, Y, epochs=300, verbose=2)

In [ ]:
# 학습과정 살펴보기(시각화)
import matplotlib.pyplot as plt
fig, loss_ax = plt.subplots(figsize=(15,6))
loss_ax.plot(hist.history['loss'], 'y', label='train_loss')
acc_ax = loss_ax.twinx()
acc_ax.plot(hist.history['accuracy'], 'g', label='train_accuracy')

acc_ax.set_ylabel('accuracy')
loss_ax.set_xlabel('epoch')
loss_ax.set_ylabel('loss')
loss_ax.legend(bbox_to_anchor=(0.97,0.75))
acc_ax.legend(loc='center right')
plt.show()

In [ ]:
# 모델 사용하기(경마장에 있는 -> 말이)
from tensorflow.keras.preprocessing.sequence import pad_sequences
encoded = t.texts_to_sequences(['6시가 되면'])[0]
input_data = pad_sequences([encoded], maxlen=5, padding='pre')
print('모델에 들어갈 입력 데이터 :',input_data)
result = model.predict(input_data).argmax(axis=1)[0]
print('모델 예측 결과 :', result)
for key, value in t.word_index.items():
    if value==result:
        print('가장 확률이 높은 단어(예측된 단어) :', key)
        break

# 2. 다음 문맥 예측해 보기

In [ ]:
# '가는 말이' 이후에 올 단어 4개 예측
def sentence_generation(model, t, current_word, n):
    print('입력된 단어 :', current_word)
    for i in range(1, n+1):
        encoded = t.texts_to_sequences([current_word])[0]
        input_data = pad_sequences([encoded], maxlen=5, padding='pre')
        result = model.predict(input_data, verbose=0).argmax(axis=1)[0]
        for key, value in t.word_index.items():
            if value==result:
                print(f'{i}번째 {key}:{result}')
                current_word = current_word + ' ' + key
                break
    return current_word

In [ ]:
sentence_generation(model, t, '경마장에', 4)